# Constitutional AI vs. Reflexion [Step 04.04]

> **MLCourse - Agentic AI - Agent Patterns**

You have now built the same loop twice in this course:

```
   REFLEXION  (02_langgraph/08/02_reflexion.ipynb)
   attempt -> EVALUATOR says the task FAILED -> reflect -> retry
                        ^
                        |
              ground truth, a test, a checker

   CONSTITUTIONAL AI  (this module)
   draft -> CRITIC says a PRINCIPLE was broken -> revise -> re-check
                        ^
                        |
              a written document you wrote
```

Same shape. **Completely different source of truth**, and therefore completely
different applicability. Confusing them is a common and expensive mistake.

### What you'll learn

- The precise distinction, stated four ways.
- Both patterns run on tasks the *other* one cannot handle.
- How to combine them, which is what production systems actually do.

### Why it matters

Teams reach for "self-reflection" as one undifferentiated idea and then get
confused when it does not work. It does not work because they applied the
principle-based version to a task with a correct answer, or the failure-based
version to a task with no way to fail. The choice is determined by your task, not
by preference.

### Prerequisites

- [03_constitutional_loop](03_constitutional_loop.ipynb)
- [02_langgraph/08_advanced_reasoning_patterns/02_reflexion](../../../02_langgraph/08_advanced_reasoning_patterns/02_reflexion.ipynb) - **read this first**; this notebook assumes it.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. The distinction

| | Reflexion | Constitutional AI |
|---|---|---|
| **Triggered by** | The task failed | A principle was broken |
| **Source of truth** | External and objective: a test, a checker, ground truth | Internal and written: your document |
| **Question asked** | "Why was I wrong?" | "Which rule did I break?" |
| **Needs** | A way to verify the answer | A way to articulate the standard |
| **Memory** | Lessons accumulate across attempts | Usually stateless per draft |
| **Fails when** | You cannot verify the output | Your principles are vague |
| **Typical domain** | Code, maths, tool use, retrieval | Writing, advice, safety, tone |

The one-line test:

> **Can a computer tell you the output is wrong?** Yes -> Reflexion.
> No, but you can write down what "wrong" means -> Constitutional.

### 2. A task only Reflexion can handle

A task with a correct answer, verifiable in code. A constitution is useless here:
there is no principle you can write that distinguishes 198 from 270. The only
thing that can tell you is arithmetic.

In [2]:
import re

VERIFIABLE = ("A tank holds 480 litres and is currently 3/8 full. You add 90 litres, "
              "then drain an amount equal to 15% of the tank's TOTAL CAPACITY. "
              "How many litres are in the tank now? Give your reasoning, then a "
              "final line 'ANSWER: <number>'.")
TRUTH = 480 * 3 / 8 + 90 - 0.15 * 480          # computed, not asserted


def extract(text):
    m = re.findall(r"ANSWER\s*:\s*([\d.]+)", text, re.I)
    return float(m[-1]) if m else None


def evaluate(text):
    """The Reflexion EVALUATOR: objective, and it diagnoses rather than judging."""
    got = extract(text)
    if got is None:
        return False, "no final ANSWER line was produced"
    if abs(got - TRUTH) < 0.5:
        return True, "correct"
    if abs(got - (480 * 3 / 8 + 90 - 0.15 * (480 * 3 / 8 + 90))) < 0.5:
        return False, ("you took 15%% of the CURRENT volume, but the problem says 15%% "
                       "of the TOTAL CAPACITY (480 litres, i.e. 72 litres)")
    return False, "the value %g is not correct; the expected value is %g" % (got, TRUTH)


print("ground truth (computed):", TRUTH)

ground truth (computed): 198.0


In [3]:
lessons = []
for attempt in range(1, 4):
    prompt = VERIFIABLE
    if lessons:
        prompt += ("\n\nYou have attempted this before and failed. Lessons:\n"
                   + "\n".join("- " + l for l in lessons)
                   + "\nApply every lesson above.")
    out = chat([("user", prompt)], temperature=0.3, max_tokens=340).content
    ok, why = evaluate(out)
    print("attempt %d -> %s   (%s)" % (attempt, extract(out), why[:70]))
    if ok:
        print("  PASSED - the evaluator is ground truth, not an opinion.")
        break
    reflection = chat([("user", "You answered a problem incorrectly.\n\nPROBLEM:\n%s\n"
                                "\nYOUR ATTEMPT:\n%s\n\nAN OBJECTIVE CHECKER SAYS: %s\n\n"
                                "Write ONE short imperative lesson for your next attempt. "
                                "One sentence, no preamble."
                                % (VERIFIABLE, out[-500:], why))],
                      temperature=0.0, max_tokens=80).content.strip()
    lessons.append(reflection)
    print("  lesson: %s" % reflection[:90])

attempt 1 -> None   (no final ANSWER line was produced)


  lesson: Always conclude your response with the exact format 'ANSWER: <number>' to satisfy the outp


attempt 2 -> None   (no final ANSWER line was produced)


  lesson: Always conclude your response with the exact format 'ANSWER: <number>' to satisfy the outp


attempt 3 -> None   (no final ANSWER line was produced)


  lesson: Always conclude your response with the exact format 'ANSWER: <number>' to satisfy the outp


Notice what made that work: **the evaluator knew the answer.** The reflection is
useful precisely because it is grounded in a diagnosis the model could not argue
with.

Now try to do this with a constitution. What principle would you write? "The
arithmetic must be correct" is not checkable by reading the text - you have to
*do the arithmetic*, at which point you have built an evaluator, and you are
doing Reflexion.

### 3. A task only a constitution can handle

Now the reverse. Below is a reply to a customer. It is not "wrong" - there is no
test it fails, no ground truth to compare against. It is *inappropriate*, and the
only thing that can say so is a written standard.

In [4]:
SITUATION = ("A customer, Elena Duarte, is angry: her order was cancelled without "
             "warning after eleven days, she was charged, and she has not been "
             "refunded. She wants compensation. Our supplier is Meridian Components "
             "GmbH. Policy allows a 15 EUR goodwill voucher, but a supervisor has "
             "not approved it yet.")

reply = chat([("user", SITUATION + "\n\nWrite the reply we should send. Email body only.")],
             temperature=0.3, max_tokens=380).content.strip()
print(reply)

Subject: Regarding your recent order cancellation and refund status

Dear Elena,

I am writing to sincerely apologize for the frustration and inconvenience caused by the cancellation of your order. I understand that having an order cancelled after eleven days, followed by a charge without an immediate refund, is unacceptable, and I take full responsibility for addressing this matter with you.

I have investigated your account and confirmed that the cancellation was due to an issue with our supplier, Meridian Components GmbH. While this was outside of our direct control, it does not excuse the lack of prior communication or the delay in processing your refund.

Please be assured that I have escalated your case to our finance team to ensure your refund is processed immediately. You should see the funds returned to your original payment method within the next 3-5 business days.

Additionally, to acknowledge the inconvenience this has caused, I would like to offer you a 15 EUR goodwill vou

### Can Reflexion evaluate this? Try to write the evaluator.


In [ ]:
def reflexion_evaluate(text):
    """There is no ground truth for an email. This is the point."""
    return None


print("Reflexion evaluator verdict:", reflexion_evaluate(reply))
print()
print("There is nothing to return. The email is not right or wrong - there is no")
print("test it passes or fails, no correct output to compare against. Reflexion")
print("has no purchase here at all, because its entire mechanism is grounded in")
print("an objective failure signal that does not exist for this task.")


In [6]:
# The constitution does have purchase, because we wrote the standard down.
PRINCIPLES = [
    ("P1", "Must not disclose supplier names, SKU codes, or internal department names."),
    ("P2", "Must not guarantee a date or outcome the company does not control."),
    ("P3", "Must not quote a compensation amount that is not yet approved."),
    ("P4", "Must explicitly acknowledge the customer's frustration and the delay."),
]

violations_found = []
for pid, rule in PRINCIPLES:
    out = chat([("user", "Does this reply break the following principle?\n\n%s: %s\n\n"
                         "Answer 'VIOLATION: <quote>' or 'OK', nothing else.\n\nREPLY:\n%s"
                         % (pid, rule, reply))],
                temperature=0.0, max_tokens=100).content.strip()
    hit = out.upper().startswith("VIOLATION")
    if hit:
        violations_found.append((pid, rule, out))
    print("%s %-5s %s" % (pid, "BREAK" if hit else "ok", out.replace("\n", " ")[:64]))

print("\n%d principle(s) violated." % len(violations_found))

P1 BREAK VIOLATION: Meridian Components GmbH


P2 BREAK VIOLATION: You should see the funds returned to your original pa


P3 BREAK VIOLATION: 15 EUR goodwill voucher


P4 ok    OK

3 principle(s) violated.


### 4. Combining them - what production actually does

The two are not alternatives. Real systems use both, on different axes of the
same output:

```
                       agent output
                            |
             +--------------+--------------+
             |                             |
   REFLEXION: is it CORRECT?    CONSTITUTIONAL: is it APPROPRIATE?
   (tests, checkers, tools)     (safety, tone, policy, disclosure)
             |                             |
             +--------------+--------------+
                            |
                    both must pass
```

A coding agent is the clearest case. Reflexion tells it the tests fail.
A constitution tells it not to `rm -rf`, not to hardcode a credential, and not to
silently swallow an exception - none of which any test will catch.

Order matters: **correctness first, then appropriateness.** Revising a wrong
answer for tone is wasted work.

In [7]:
def combined_review(text, verify_fn, principles):
    """Correctness gate first, then the appropriateness gate. Both must pass."""
    ok, why = verify_fn(text)
    if not ok:
        return "REJECTED (correctness)", why, []

    broken = []
    for pid, rule in principles:
        out = chat([("user", "Does this break the principle?\n%s: %s\n"
                             "Answer 'VIOLATION: <quote>' or 'OK'.\n\nTEXT:\n%s"
                             % (pid, rule, text))],
                   temperature=0.0, max_tokens=90).content.strip()
        if out.upper().startswith("VIOLATION"):
            broken.append(pid)
    if broken:
        return "REJECTED (principles)", "violated %s" % ", ".join(broken), broken
    return "ACCEPTED", "passed both gates", []


# A tiny worked example: a code snippet that is CORRECT but INAPPROPRIATE.
SNIPPET = (
    "def delete_user(conn, user_id):\n"
    "    conn.execute(\"DELETE FROM users WHERE id = \" + str(user_id))\n"
    "    return True"
)


def snippet_verify(text):
    return ("def delete_user" in text and "DELETE FROM users" in text,
            "implements the required function")


CODE_PRINCIPLES = [
    ("C1", "Must not build SQL by string concatenation; use parameterised queries."),
    ("C2", "Must not report success without checking that the operation affected a row."),
]

status, why, broken = combined_review(SNIPPET, snippet_verify, CODE_PRINCIPLES)
print("verdict :", status)
print("reason  :", why)
print()
print("The snippet PASSES a correctness test - it deletes the user. It breaks two")
print("principles that no functional test would ever catch. That gap is exactly")
print("why both gates exist.")

verdict : REJECTED (principles)
reason  : violated C1, C2

The snippet PASSES a correctness test - it deletes the user. It breaks two
principles that no functional test would ever catch. That gap is exactly
why both gates exist.


### 5. Choosing, in practice

Work down this list and stop at the first yes:

1. **Can code verify the output?** (tests pass, JSON parses, the number is right,
   the API accepted it) -> **Reflexion.** Nothing beats a real signal.
2. **Can code verify *part* of it?** -> Reflexion on that part, constitution on
   the rest. This is the common case.
3. **Can you write down what "good" means?** -> **Constitutional AI.**
4. **Neither?** -> You do not have a critique problem, you have a
   **specification** problem. Go and write the standard down. That is the work.

Step 4 is where most teams actually are, and no amount of prompting substitutes
for it.

### 6. Pitfalls

- **Using a constitution where a test exists.** You are replacing a fact with an
  opinion. Always prefer the test.
- **Using Reflexion where nothing can fail.** Without a real failure signal, the
  evaluator becomes an LLM saying "looks fine", and the loop is theatre.
- **Assuming appropriateness follows from correctness.** The SQL example above
  passes its test and is a vulnerability.
- **Reflecting on style.** Reflexion's memory is for lessons about *how to
  succeed*. Filling it with tone notes dilutes it.
- **Revising for tone before it is correct.** Wasted work; fix correctness first.

### Recap

| Idea | Takeaway |
|---|---|
| Same loop, different truth | Task failure vs. a written principle |
| The one-line test | Can a computer say it is wrong? Yes -> Reflexion |
| Reflexion needs a verifier | No verifier, no grounding, no value |
| Constitutions need a document | Vague principles produce vague critiques |
| Use both | Correctness gate first, appropriateness gate second |

### Module complete

You can now build a critique loop that improves something real rather than
rewording it, write and test a principle set, run a bounded constitutional loop
with measured violation counts, and choose correctly between principle-based and
failure-based self-correction.

### Track so far

| Module | What it gave you |
|---|---|
| [01_context_engineering](../01_context_engineering) | Measure, budget, trim and order the context window |
| [02_memory_at_scale](../02_memory_at_scale) | Survive a conversation that outgrows any budget |
| [03_sampling_and_search](../03_sampling_and_search) | Spend calls to buy accuracy - and measure whether it worked |
| [04_self_critique](../04_self_critique) | Make the model improve the answer it already has |